Для разметки будем исползовать - Qwen2.5-VL-7B-Instruct

### Стратегия Разметки

Мы создадим **три типа вопросов**:

1.  **Text-Only QA (Теория):**
    *   *Вход:* Чанк текста из `theory.json`.
    *   *Промпт:* "Сформулируй технический вопрос, ответ на который содержится в этом тексте. И дай ответ."
2.  **Visual-Only QA (Топология):**
    *   *Вход:* Картинка схемы из `images.json`.
    *   *Промпт:* "Посмотри на схему. Какой элемент подключен к...? / Какого типа источник?"
3.  **Multimodal QA (Reasoning):**
    *   *Вход:* Картинка + Текст, который был перед ней.
    *   *Промпт:* "Используя схему и описание, объясни принцип работы..."




In [1]:
import os
import torch
BASE_DIR = os.environ.get("EE_RAG_BASE_DIR", os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"Данные: {DATA_DIR}")
print(f"Устройство: {DEVICE}")


BASE_DIR: /multimodal-rag-article
Данные: /multimodal-rag-article/data
Устройство: cuda


Загрузка модели (Qwen2.5-VL-7B-Instruct)

In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
torch_dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    attn_implementation="sdpa",
    device_map="auto" if DEVICE == "cuda" else None
)
if DEVICE != "cuda":
    model = model.to(DEVICE)
processor = AutoProcessor.from_pretrained(model_name)


/opt/conda/envs/qwen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 729/729 [00:46<00:00, 15.59it/s] 
/opt/conda/envs/qwen/lib/python3.11/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/blobs only has 0.00 MB free disk space.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/opt/conda/envs/qwen/lib/python3.

Пишем промпты для разных типов данных. Модель будет возвращать json

In [3]:
import json
import random
from PIL import Image
def generate_qa_pair(item, mode="text", preceding_text=None):
    """
    mode: 'text', 'image', 'hybrid'
    """
    if mode == "text":
        user_content = [
            {"type": "text", "text": f"Текст учебника:\n{item['text']}\n\nЗАДАЧА: Придумай 1 конкретный вопрос по электротехнике, на который можно ответить, опираясь ТОЛЬКО на этот текст. Не используй общие фразы. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]
    elif mode == "image":
        user_content = [
            {"type": "image", "image": item['path']},
            {"type": "text", "text": "Это электрическая схема. Внимательно проанализируй её топологию (связи элементов). Придумай 1 вопрос о том, как соединены элементы или что изображено. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]
    elif mode == "hybrid":
        # Склеиваем описание и картинку
        context_text = preceding_text if preceding_text else "Описание отсутствует."
        user_content = [
            {"type": "image", "image": item['path']},
            {"type": "text", "text": f"Текст описания:\n{context_text}\n\nЗАДАЧА: Используя И схему, И текст, придумай сложный вопрос на понимание. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]
    messages = [{"role": "user", "content": user_content}]
    # 3. Инференс
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    # Генерация
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    if "assistant\n" in output_text:
        output_text = output_text.split("assistant\n")[-1]
    # Попытка парсинга JSON
    try:
        # Убираем markdown обертку ```json ... ```
        clean_json = output_text.replace("```json", "").replace("```", "").strip()
        qa = json.loads(clean_json)
        return qa
    except:
        return None



Тестовый прогон на 10 семплах:

In [4]:
import random
import json

with open(os.path.join(DATA_DIR, "theory.json"), 'r') as f:
    theory_data = json.load(f)
with open(os.path.join(DATA_DIR, "images.json"), 'r') as f:
    images_data = json.load(f)

text_lookup = {t['id']: t['text'] for t in theory_data}

samples_text = random.sample(theory_data, 5)
valid_images = [img for img in images_data if os.path.exists(img['path'])]
samples_img = random.sample(valid_images, 5)

test_dataset = []


print("Generating Text Questions")
for item in samples_text:
    qa = generate_qa_pair(item, mode="text")
    if qa:
        entry = {
            "question": qa['question'],
            "answer": qa['answer'],
            "type": "text",
            "gold_ids": [item['id']], # Правильный ответ - этот чанк текста
            "metadata": {
                "source_url": item.get('source_url', ''),
                "lecture": item.get('lecture_id', '')
            }
        }
        test_dataset.append(entry)
        print(f"Text QA: {qa['question'][:50]}...")

print("Generating Hybrid Questions")
for item in samples_img:
    prec_text_id = item.get('preceding_text_id')
    prec_text = text_lookup.get(prec_text_id, "")

    # Эмулируем случайный выбор режима (как в проде)
    mode = "image" if random.random() < 0.3 else "hybrid"

    qa = generate_qa_pair(item, mode=mode, preceding_text=prec_text)
    if qa:
        # Собираем ID
        gold_ids = [item['id']]
        if mode == "hybrid" and prec_text_id:
            gold_ids.append(prec_text_id) # Текст тоже правильный ответ

        entry = {
            "question": qa['question'],
            "answer": qa['answer'],
            "type": mode,
            "gold_ids": gold_ids,
            "metadata": {
                "image_path": item['path'],
                "caption": item.get('caption', '')
            }
        }
        test_dataset.append(entry)
        print(f"{mode.upper()} QA: {qa['question'][:50]}")

print(f"Успешно сгенерировано: {len(test_dataset)}/10")
print("Пример финального JSON (проверьте структуру):")
print(json.dumps(test_dataset[-1], indent=2, ensure_ascii=False))

Generating Text Questions
Text QA: Какую информацию можно получить о топологии электр...
Text QA: Какое значение нагрузки RН позволяет источнику пот...
Text QA: Какие параметры ЭДС и напряжения определяются анал...
Generating Hybrid Questions
IMAGE QA: Какие элементы соединены в этой схеме?
IMAGE QA: Какие элементы соединены в этой схеме?
HYBRID QA: Какое количество независимых уравнений по второму 
Успешно сгенерировано: 6/10
Пример финального JSON (проверьте структуру):
{
  "question": "Какое количество независимых уравнений по второму закону Кирхгофа можно составить для схемы с n ветвями, если известно, что m из них являются ветвями дерева?",
  "answer": "n - m + 1",
  "type": "hybrid",
  "gold_ids": [
    "L02_p55_img0",
    "L02_p55"
  ],
  "metadata": {
    "image_path": "data/images/L02_p55_img0.png",
    "caption": ""
  }
}


In [5]:
test_dataset

[{'question': 'Какую информацию можно получить о топологии электрической цепи, зная только одну из топологических матриц?',
  'answer': 'Из одной топологической матрицы можно восстановить остальные матрицы, что позволяет определить структуру и взаимосвязи компонентов электрической цепи.',
  'type': 'text',
  'gold_ids': ['L02_p74'],
  'metadata': {'source_url': 'https://www.toehelp.ru/theory/toe/lecture02/lecture02.html',
   'lecture': 'L02'}},
 {'question': 'Какое значение нагрузки RН позволяет источнику потреблять максимальную мощность?',
  'answer': 'Согласованный режим работы',
  'type': 'text',
  'gold_ids': ['L01_p55'],
  'metadata': {'source_url': 'https://www.toehelp.ru/theory/toe/lecture01/lecture01.html',
   'lecture': 'L01'}},
 {'question': 'Какие параметры ЭДС и напряжения определяются аналогично?',
  'answer': 'Аналогично определяются действующие значения ЭДС и напряжения.',
  'type': 'text',
  'gold_ids': ['L03_p16'],
  'metadata': {'source_url': 'https://www.toehelp.ru/t

Финальный запуск и генерация датасета для оцеки

In [8]:
%%time
from tqdm.notebook import tqdm
import random
import json
import time
SEED = int(os.environ.get("BENCHMARK_SEED", "42"))
TARGET_COUNT = int(os.environ.get("BENCHMARK_TARGET_COUNT", "1000"))
BATCH_SAVE = int(os.environ.get("BENCHMARK_BATCH_SAVE", "10"))
OUTPUT_FILE = os.environ.get("BENCHMARK_OUTPUT_FILE", os.path.join(DATA_DIR, "benchmark_final.json"))
random.seed(SEED)
print(f"SEED: {SEED}")
print(f"TARGET_COUNT: {TARGET_COUNT}")
print(f"OUTPUT_FILE: {OUTPUT_FILE}")
# Загрузка
with open(os.path.join(DATA_DIR, "theory.json"), 'r') as f:
    theory_data = json.load(f)
with open(os.path.join(DATA_DIR, "images.json"), 'r') as f:
    images_data = json.load(f)
text_lookup = {t['id']: t['text'] for t in theory_data}
valid_images = [img for img in images_data if os.path.exists(img['path'])]
# Перемешиваем
random.shuffle(theory_data)
random.shuffle(valid_images)
gold_dataset = []
cnt_text = 0
cnt_visual = 0
print(f"Начинаем СМЕШАННУЮ генерацию {TARGET_COUNT} примеров...")
for i in range(TARGET_COUNT):
    print(f"{i+1}/{TARGET_COUNT} (Txt:{cnt_text} Img:{cnt_visual}) ...", end="")
    try:
        entry = None
        target_text = TARGET_COUNT // 2
        # Определяем, что генерить сейчас (бросаем монетку, но следим за лимитами)
        want_text = (random.random() < 0.5)
        # Если выпал текст И лимит не исчерпан ИЛИ картинки кончились -> делаем Текст
        if (want_text and cnt_text < target_text) or (cnt_visual >= len(valid_images)):
            if cnt_text < len(theory_data):
                item = theory_data[cnt_text]
                cnt_text += 1
                qa = generate_qa_pair(item, mode="text")
                if qa:
                    entry = {
                        "question": qa['question'], "answer": qa['answer'],
                        "type": "text", "gold_ids": [item['id']],
                        "metadata": {"source_id": item['id']}
                    }
        # Иначе -> делаем Картинку
        else:
            if cnt_visual < len(valid_images):
                item = valid_images[cnt_visual]
                cnt_visual += 1
                prec_text_id = item.get('preceding_text_id')
                prec_text = text_lookup.get(prec_text_id, "")
                mode = "image" if random.random() < 0.3 else "hybrid"
                qa = generate_qa_pair(item, mode=mode, preceding_text=prec_text)
                if qa:
                    gold_ids = [item['id']]
                    if mode == "hybrid" and prec_text_id: gold_ids.append(prec_text_id)
                    entry = {
                        "question": qa['question'], "answer": qa['answer'],
                        "type": mode, "gold_ids": gold_ids,
                        "metadata": {"image_path": item['path']}
                    }
        # Обработка результата
        if entry:
            gold_dataset.append(entry)
            # Выводим тип вопроса, чтобы видеть баланс
            print(f"[{len(gold_dataset)}] {entry['type'].upper()}: {entry['question'][:40]}...")
        else:
            pass
    except Exception as e:
        print(f"Error: {e}")
        continue
    # Сохранение
    if len(gold_dataset) % BATCH_SAVE == 0:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(gold_dataset, f, ensure_ascii=False, indent=4)
# Финал
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(gold_dataset, f, ensure_ascii=False, indent=4)
print(f"ГОТОВО! Собрано: {len(gold_dataset)}")


SEED: 42
TARGET_COUNT: 1000
OUTPUT_FILE: /multimodal-rag-article/data/benchmark_final.json
Начинаем СМЕШАННУЮ генерацию 1000 примеров...
1/1000 (Txt:0 Img:0) ...2/1000 (Txt:0 Img:1) ...3/1000 (Txt:0 Img:2) ...4/1000 (Txt:0 Img:3) ...[1] HYBRID: Как определить начальную фазу синусоидал...
5/1000 (Txt:0 Img:4) ...[2] TEXT: Какие компоненты в уравнениях закона Кир...
6/1000 (Txt:1 Img:4) ...7/1000 (Txt:2 Img:4) ...[3] TEXT: Какой метод расчета цепей переменного си...
8/1000 (Txt:3 Img:4) ...[4] HYBRID: Как метод кусочно-линейной аппроксимации...
9/1000 (Txt:3 Img:5) ...[5] TEXT: Какие методы могут быть использованы для...
10/1000 (Txt:4 Img:5) ...[6] TEXT: Какой параметр характеризует сопротивлен...
11/1000 (Txt:5 Img:5) ...[7] HYBRID: Какой принцип используется для анализа п...
12/1000 (Txt:5 Img:6) ...[8] HYBRID: Какое значение тока Ii следует задать на...
13/1000 (Txt:5 Img:7) ...[9] HYBRID: Как изменится значение радиуса расточки ...
14/1000 (Txt:5 Img:8) ...15/1000 (Txt:6 Img:8) ...[